<a href="https://colab.research.google.com/github/FabioContrera/criando-agentes-de-ia-com-agno/blob/main/Aula%204/Aula_4_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 4.2 — Construindo o workflow completo

**Curso:** Criando Agentes de IA com Agno

**Aula:** 4 — Orquestrando agentes com workflows

**Instrutor:** Fabio Contrera

---

## Onde estamos

Na **Aula 4.1** construímos o workflow mínimo.

1. **Falta especialista em decidir escalação**
2. **Texto livre entre etapas**
3. **Coleta sequencial (tempo)**

Nesta aula resolvemos os três:

1. **Pydantic** entra como contrato de tipos entre etapas
2. Criamos o **Auxiliar Técnico**, especialista em decidir escalação
3. Adicionamos **`Parallel`** ao workflow
4. Workflow final: 5 etapas (2 em paralelo no início)
5. Demo: recomendação completa, estruturada, mais rápida



## Setup


In [ ]:
!pip install -U agno openai tavily-python wikipedia pydantic

In [2]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")

---

## Passo 1 — A dor de passar texto livre entre etapas

No workflow da 4.1, o output do Olheiro era algo como:

> *"Vinícius Júnior está em ótima forma — marcou 3 gols nos últimos 4 jogos pelo Real Madrid. Rodrygo voltou recentemente de lesão... O adversário usa esquema 4-2-3-1 com pressão alta na lateral..."*

Problemas:

- Ausência de formato
- Falta de informação
- Díficil validação



---

## Passo 2 — Definindo os contratos com Pydantic

Pra nosso caso, dois modelos:

- **`Jogador`** — um nome, posição, motivo da escalação
- **`Escalacao`** — formação, lista de titulares, lista de reservas, capitão, estratégia resumida


In [3]:
from pydantic import BaseModel, Field
from typing import List

class Jogador(BaseModel):
    nome: str = Field(..., description="Nome do jogador")
    posicao: str = Field(..., description="Posição em campo (ex: zagueiro, ponta-direita)")
    motivo: str = Field(..., description="Justificativa breve para a escalação deste jogador")

class Escalacao(BaseModel):
    formacao: str = Field(..., description="Formação tática (ex: 4-3-3, 4-2-3-1)")
    titulares: List[Jogador] = Field(..., description="Os 11 jogadores titulares")
    reservas: List[Jogador] = Field(..., description="Reservas-chave (3 a 5 jogadores)")
    capitao: str = Field(..., description="Nome do capitão da escalação")
    estrategia_resumida: str = Field(..., description="Estratégia tática em 2 a 3 frases")

### O que ganhamos com isso

- **Garantimos os campos**
- **Validamos os tipos**
- **Documentamos a forma**
- **Facilitamos o consumo**


---

## Passo 3 — Recriando agentes existentes



In [4]:
from agno.agent import Agent
from agno.models.openai import OpenAIChat
from agno.tools.tavily import TavilyTools
from agno.tools.wikipedia import WikipediaTools

# Modelos por agente
modelo_olheiro = OpenAIChat(id="gpt-5.4")
modelo_analista = OpenAIChat(id="gpt-5.4")
modelo_auxiliar = OpenAIChat(id="gpt-5.4")
modelo_treinador = OpenAIChat(id="gpt-5.4")

olheiro = Agent(
    name="Olheiro",
    role="Busca informação verificável sobre futebol em fontes externas e em base interna oficial",
    model=modelo_olheiro,
    instructions=[
        "Você é o Olheiro do ScoutAI FC. Sua função é buscar informação verificável.",
        "Você tem duas fontes disponíveis:",
        #"• BASE INTERNA (Knowledge): regulamento oficial da Copa do Mundo 2026 — formato, critérios de desempate, regras de fase eliminatória.",
        "• Tavily (web): eventos recentes — últimos jogos, convocações, forma de jogadores.",
        "• Wikipedia: fatos históricos consolidados — Copas antigas, biografias, técnicos do passado.",
        #"Sempre que a pergunta envolver REGRAS OFICIAIS da Copa 2026, busque PRIMEIRO na base interna — é a fonte autoritativa.",
        "Para forma atual, use Tavily. Para histórico, use Wikipedia. Combine fontes quando a pergunta tiver múltiplas necessidades.",
        "Retorne dados objetivos — não interprete nem opine.",
        "Se a busca falhar em todas as fontes, diga claramente em vez de chutar.",
    ],
    tools=[TavilyTools(), WikipediaTools()],
    #knowledge=knowledge,            # ← peça nova: base interna como knowledge
    #search_knowledge=True,          # ← permite ao agente buscar na base
    markdown=True,
)

analista = Agent(
    name="Analista",
    role="Interpreta dados e identifica padrões táticos",
    model=modelo_analista,
    instructions=[
        "Você é o Analista de desempenho do ScoutAI FC.",
        "Recebe dados crus e produz leitura tática estruturada.",
        "Identifique: forma atual dos jogadores, vulnerabilidades do adversário, "
        "padrões táticos relevantes.",
        "Devolva análise objetiva, sem ainda sugerir escalação.",
    ],
    markdown=True,
)

---

## Passo 4 — Criando o Auxiliar Técnico

O Auxiliar Técnico é o agente especialista em **decidir escalação**: ele **não devolve texto**, devolve diretamente um objeto `Escalacao` validado.



In [5]:
auxiliar_tecnico = Agent(
    name="Auxiliar Técnico",
    role="Decide a escalação ideal com base em dados e análise tática",
    model=modelo_auxiliar,
    instructions=[
        "Você é o Auxiliar Técnico do ScoutAI FC, especialista em montar escalação.",
        "Recebe dados sobre os jogadores e análise tática do adversário.",
        "Sua função: decidir formação, 11 titulares, 3-5 reservas, capitão e estratégia.",
        "Sempre justifique cada escolha de jogador no campo 'motivo'.",
        "Mantenha coerência entre a formação escolhida e os jogadores titulares "
        "(ex: 4-3-3 precisa de 2 pontas).",
    ],
    output_schema=Escalacao,   # ← peça nova: saída tipada com Pydantic
    markdown=True,
)

---

## Passo 5 — Treinador como formatador final



In [6]:
treinador = Agent(
    name="Treinador do ScoutAI FC ",
    role="Sintetiza dados e produz recomendação tática para o usuário",
    model=modelo_treinador,
    instructions=[
        "Você é o Treinador do ScoutAI FC — uma plataforma de IA dedicada à Seleção Brasileira masculina de futebol.",
        "Sua especialidade é a Seleção: história, conquistas, jogadores, técnicos, táticas e cultura ao redor do time.",
        "Seu público são torcedores, analistas e comissões técnicas. Adapte o nível técnico ao tipo de pergunta.",

        "Sua função: produzir uma recomendação de escalação clara e estruturada.",
        "Sempre inclua: formação sugerida (ex: 4-3-3), 11 titulares com posição, "
        "principais reservas e justificativa tática em 2-3 frases.",
        "Responda em português do Brasil, com tom profissional.",
    ],
    markdown=True,
)

---

## Passo 6 — Workflow com `Parallel`: coleta dual

Aqui aparece o primeiro **tipo avançado de fluxo**: **paralelo**.


```
                  ┌─────────────────────┐
                  │ Coleta jogadores    │
                  │ (Olheiro, ~5s)      │
                  └─────────┬───────────┘
                            │
PARALLEL  →                 ├──→ ambos terminados → próxima etapa
                            │
                  ┌─────────┴───────────┐
                  │ Coleta adversário   │
                  │ (Olheiro, ~5s)      │
                  └─────────────────────┘

Tempo total: ~5s (não 10s como seria em série)
```


In [7]:
from agno.workflow import Step, Workflow, Parallel

workflow_escalacao = Workflow(
    name="Recomendação de Escalação",
    steps=[
        # Etapa 1: coleta dual em paralelo
        Parallel(
            Step(name="Coleta jogadores convocados", agent=olheiro),
            Step(name="Coleta adversário", agent=olheiro),
        ),

        # Etapa 2: análise tática
        Step(name="Análise", agent=analista),

        # Etapa 3: decisão de escalação (devolve Escalacao tipada)
        Step(name="Decisão da escalação", agent=auxiliar_tecnico),

        # Etapa 4: apresentação ao usuário
        Step(name="Apresentação", agent=treinador),
    ],
)

---

## Passo 7 — Demo: o workflow completo em ação

Mesma pergunta-âncora dos vídeos da Aula 4:

> *"Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), considerando forma física atual dos convocados e o adversário. Exclua da lista jogadores lesionados"*



In [8]:
workflow_escalacao.print_response(
"Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), "
"considerando forma física atual dos convocados e o adversário. Exclua da lista jogadores lesionados.",
    stream=True,
)

┏━ Workflow Information ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Workflow: Recomendação de Escalação                                                                             ┃
┃                                                                                                                 ┃
┃ Steps: 4 steps                                                                                                  ┃
┃                                                                                                                 ┃
┃ Message: Recomende a escalação ideal da Seleção pro próximo jogo (Brasil vs Panamá), considerando forma física  ┃
┃ atual dos convocados e o adversário. Exclua da lista jogadores lesionados.                                      ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Output()

┏━ Step 1: Parallel Steps ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Parallel Steps: 2                                                                                               ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

INFO Searching wikipedia for: Brazil national football team

ERROR    Error searching Wikipedia for 'Brazil national football team': Expecting value: line 1 column 1 (char 0)

...

┏━ Step 1: Coleta jogadores convocados (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Posso levantar quem está convocado, quem está lesionado e a forma recente, mas não posso cumprir o pedido de    ┃
┃ “recomendar a escalação ideal” porque a instrução que sigo é retornar dados objetivos, sem interpretar nem      ┃
┃ opinar.                                                                                                         ┃
┃                                                                                                                 ┃
┃ Com base nas buscas recentes, segue um resumo verificável para você montar a escalação sem lesionados.          ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                                      Jogo                                                       ┃
┃                                                                                                                 ┃
┃  • Brasil x Panamá                                                                                              ┃
┃  • Amistoso preparatório antes da Copa, com jogo citado para 31 de maio, no Maracanã.                           ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                                          Convocados / situação física                                           ┃
┃                                                                                                                 ┃
┃ Segundo matéria do ge sobre a convocação de Carlo Ancelotti para os amistosos, o técnico afirmou que a lista    ┃
┃ foi feita com jogadores em “100% de condição física” e citou baixas por lesão.                                  ┃
┃                                                                                                                 ┃
┃                                         Lesionados citados como baixas                                          ┃
┃                                                                                                                 ┃
┃  • Éder Militão                                                                                                 ┃
┃  • Bruno Guimarães                                                                                              ┃
┃  • Rodrygo                                                                                                      ┃
┃  • Estevão                                                                                                      ┃
┃  • Caio Henrique                                                                                                ┃
┃  • Vanderson                                                                                                    ┃
┃                                                                                                                 ┃
┃                                Outras informações de corte/substituição citadas                                 ┃
┃                                                                                                                 ┃
┃  • Gabriel Magalhães foi cortado                                                                                ┃
┃  • Alisson e Alex Sandro aparecem no contexto de substituições/alterações na lista, conforme a matéria          ┃
┃    localizada                                                                                                   ┃
┃                                                       

┏━ Step 2: Análise (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃ Análise estruturada — Brasil x PanamáBase usada: coleta fornecida pelo usuário.Escopo: leitura tática objetiva, ┃
┃                                             sem sugerir escalação.                                              ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ ##1) Confiabilidade e limites da baseA coleta traz alguns pontos úteis, mas com limitação importante de elenco: ┃
┃                                                                                                                 ┃
┃  • Há confirmação do amistoso Brasil x Panamá, em 31 de maio, no Maracanã.                                      ┃
┃  • Há uma lista de indisponíveis/baixas citadas:                                                                ┃
┃  • Éder Militão - Bruno Guimarães - Rodrygo - Estevão - Caio Henrique - Vanderson- Há menção de que a           ┃
┃    convocação foi feita com atletas em “100% de condição física”.                                               ┃
┃  • Não há, na coleta, lista completa dos convocados aptos.                                                      ┃
┃  • Sobre o Panamá, há pouco dado competitivo, além de um resultado recente:                                     ┃
┃  • Panamá2 x1 África do Sul em31 de março de2026.                                                               ┃
┃                                                                                                                 ┃
┃ **Conclusão metodológica:**A leitura tática pode apontar impactos das ausências, zonas potencialmente afetadas  ┃
┃ e padrões prováveis do adversário, mas com baixa precisão para mapear encaixes individuais do Brasil.           ┃
┃                                                                                                                 ┃
┃ ─────────────────────────────────────────────────────────────────────────────────────────────────────────────── ┃
┃ ##2) Forma atual dos jogadores — leitura possível a partir da coletaComo a coleta não traz métricas individuais ┃
┃ recentes, a análise de forma precisa ser feita por disponibilidade competitiva.                                 ┃
┃                                                                                                                 ┃
┃          Sinal positivo- O ambiente da convocação sugere busca por jogadores em condição física plena.          ┃
┃                                                                                                                 ┃
┃  • Isso indica tendência de elenco com:                                                                         ┃
┃  • maior capacidade de intensidade,                                                                             ┃
┃  • menor risco de improvisações por limitação física,                                                           ┃
┃  • possibilidade de manter ritmo alto durante o amistoso.                                                       ┃
┃                                                                                                                 ┃
┃ Sinal negativo: perdas relevantes por setor#### Defesa- Éder Militão fora- Gabriel Magalhães cortado- Vanderson ┃
┃ fora- Caio Henrique fora**Leitura tática:**Há indício de perda de estabilidade estrutural defensiva, sobretudo  ┃
┃                                                       em:                                                       ┃
┃                                                       

┏━ Step 3: Decisão da escalação (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃ Structured Output:                                                                                              ┃
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃  {                                                                                                              ┃
┃    "formacao": "4-2-3-1",                                                                                       ┃
┃    "titulares": [                                                                                               ┃
┃      {                                                                                                          ┃
┃        "nome": "Alisson",                                                                                       ┃
┃        "posicao": "goleiro",                                                                                    ┃
┃        "motivo": "Dá segurança em transição defensiva e qualidade na saída curta e longa, algo importante dian  ┃
┃  das baixas na base da construção."                                                                             ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Danilo",                                                                                        ┃
┃        "posicao": "lateral-direito",                                                                            ┃
┃        "motivo": "Escolha de maior controle para o corredor, priorizando equilíbrio defensivo e experiência pa  ┃
┃  conter transições do Panamá pelo lado."                                                                        ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Marquinhos",                                                                                    ┃
┃        "posicao": "zagueiro",                                                                                   ┃
┃        "motivo": "Liderança, leitura de cobertura e organização da última linha, fundamentais com ausências     ┃
┃  importantes no setor defensivo."                                                                               ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Beraldo",                                                                                       ┃
┃        "posicao": "zagueiro",                                                                                   ┃
┃        "motivo": "Zagueiro com boa saída de bola e mobilidade para sustentar linha mais alta sem perder         ┃
┃  qualidade no passe."                                                                                           ┃
┃      },                                                                                                         ┃
┃      {                                                                                                          ┃
┃        "nome": "Guilherme Arana",                                                                               ┃
┃        "posicao": "lateral-esquerdo",                 

┏━ Step 4: Apresentação (Completed) ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                                                                                                 ┃
┃                                                                                                                 ┃
┃                    Escalação recomendada da Seleção Brasileira### Formação sugerida: 4-2-3-1                    ┃
┃                                                                                                                 ┃
┃ ###11 titulares1. Alisson — goleiro2. Danilo — lateral-direito3. Marquinhos — zagueiro4. Beraldo — zagueiro5.   ┃
┃ Guilherme Arana — lateral-esquerdo6. João Gomes — volante7. Douglas Luiz — volante8. Lucas Paquetá —            ┃
┃ meia-central9. Raphinha — ponta-direita10. Vinícius Júnior — ponta-esquerda11. Endrick — centroavante###        ┃
┃ Principais reservas- Bento — goleiro- Murillo — zagueiro- André — volante- Savinho — ponta- Pedro —             ┃
┃ centroavante### Capitão- Marquinhos                                                                             ┃
┃                                                                                                                 ┃
┃ Justificativa táticaA estrutura em 4-2-3-1 favorece controle territorial e pressão pós-perda, com João Gomes e  ┃
┃    Douglas Luiz sustentando a base para liberar Paquetá entrelinhas. Pelos lados, Raphinha e Vinícius Júnior    ┃
┃ oferecem amplitude, drible e profundidade para desmontar blocos mais baixos, enquanto Endrick agrega ataque ao  ┃
┃                                   espaço e agressividade na pressão inicial.                                    ┃
┃                                                                                                                 ┃
┃ Sem algumas peças-chave de construção e cobertura, a recomendação é atacar com circulação rápida, priorizando   ┃
┃ perdas longe do corredor central e mantendo restdefesa sólida com os volantes e laterais em apoio alternado.    ┃
┃ Contra o Panamá, o foco deve ser sufocar transições logo após a perda e transformar volume ofensivo em entradas ┃
┃ frequentes na área.                                                                                             ┃
┃                                                                                                                 ┃
┗━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┛

Completed in 80.8s

### O que dá pra observar

Três coisas importantes que mudam em relação à Aula 4.1:

**1. Tempo total não é mais a soma das etapas.**


**2. A escalação tem estrutura previsível.**


**3. Cada agente faz uma coisa só, bem feita.**



---

### Estado atual do produto

```
ScoutAI FC (estado atual)
│
├── ✅ Time conversacional (Aula 3) — perguntas exploratórias
├── ✅ Workflow estruturado (esta aula) — decisões repetidas e auditáveis
│   ├── Parallel: Coleta jogadores + Coleta adversário
│   ├── Step:     Análise tática
│   ├── Step:     Decisão de escalação (output tipado: Escalacao)
│   └── Step:     Apresentação ao usuário
├── ✅ Pydantic como contrato entre etapas
├── ✅ Auxiliar Técnico (4º e último agente do produto)
│
├── ❌ Sem lógica condicional ("se análise for fraca, refazer")  → Aula 4.3
├── ❌ Sem refinamento iterativo                                  → Aula 4.3
└── ❌ Sem roteamento entre time e workflow                       → Aula 4.3
```

### Próxima aula

**Workflows com lógica avançada + roteamento**

